In [1]:

import numpy as np

from scripts.data_filter import filter_dataframe
from scripts.data_loader import load_dataframe
from scripts.utils import RF_PARAM_5G, NETWORK_TYPE

filename = "5G_data_2023.mat"

# Series of random seeds for reproducability
random_seeds = np.loadtxt("data/random_seeds.csv", dtype=int)

# load the dataframe from saved file or 'raw' matlab file
df = load_dataframe(filename, NETWORK_TYPE._5G)

# # Drop unused columns to save space
matrix_cols_to_drop = ["toa_pps", "toa_cir", "toa_cov", "campaign_id"]
df["measurements_matrix"] = df["measurements_matrix"].apply(
    lambda x: x.drop(columns=matrix_cols_to_drop)
)

operator_choice = [10]
selected_campaigns = list(range(1, 21))
rf_param = RF_PARAM_5G.RSRQ

# Data filtering
df_orig = filter_dataframe(
    df=df,
    operators=operator_choice,
    include_columns=[
        "pci",
        "beam_index",
        "nr_arfcn",
        "operator_id",
        "sinr",
        "rsrq"
    ],
    campaigns=selected_campaigns,
)


Loaded dataframe from .h5 file: /Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/data/dataframe_cache/5G_data_2023.h5


/Users/andreres/Documents/UIO/master/thesis/dev/5G_localization/scripts/data_filter.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["measurements_matrix"] = df["measurements_matrix"].apply(


In [2]:
from scripts.beamforming import get_best_beam, get_beam_sidelobe_pcis
from scripts.matrix_operations import create_point_matrix
import pandas as pd

df = df_orig.sample(10)

df["best_beam"] = df["measurements_matrix"].apply(
    lambda x: get_best_beam(x, rf_param)
)
i = 1

tp = df.iloc[i]

best_beam = tp["best_beam"]
sidelobes = get_beam_sidelobe_pcis(best_beam)

print(best_beam)
for s in sidelobes:
    print(s)

tp = pd.DataFrame([tp])

pcis = [best_beam]

ma, id = create_point_matrix(tp, pcis, rf_param)

pcis = get_beam_sidelobe_pcis(best_beam)

ma2, id2 = create_point_matrix(tp, sidelobes, rf_param)

ma

# mat = tp.iloc[0]['measurements_matrix']
#
# for i, row in mat.iterrows():
#     npc_tuple = (
#         row["pci"],
#         row["beam_index"],
#         row["nr_arfcn"],
#         row["operator_id"],
#     )
#
#     if npc_tuple == best_beam:
#         print("match at ", i)


(np.float64(-108.0), np.float64(1.0), np.float64(643296.0), np.float64(10.0))
(np.float64(-108.0), np.float64(0.0), np.float64(643296.0), np.float64(10.0))
(np.float64(-108.0), np.float64(1.0), np.float64(643296.0), np.float64(10.0))
(np.float64(-108.0), np.float64(2.0), np.float64(643296.0), np.float64(10.0))


array([[-10.33139669]])

In [3]:
ma2

array([[-10.60597169, -10.33139669, -10.49011612]])